In [2]:
import os
import pandas as pd

In [3]:
# 1. Load the file
df = pd.read_excel("News_Amar_desh.xlsx")

# 2. Keep only the two columns
df_clean = df[['title', 'news']]

# 3. Save it back
df_clean.to_excel("News_Amar_desh_cleaned.xlsx", index=False)
print("Done! Kept only title and content.")

Done! Kept only title and content.


In [5]:
def extract_enriched_identities(input_file, output_dir="extracted_identities_amar_desh"):
    print(f"Loading dataset from: {input_file}...")
    
    # 1. Load data and clean missing text rows
    try:
        df = pd.read_excel(input_file)
    except FileNotFoundError:
        print(f"Error: The file '{input_file}' was not found in the current directory.")
        return

    # Drop rows where both title and content are missing
    df_clean = df.dropna(subset=['title', 'news']).reset_index(drop=True)
    
    # Fill remaining NaNs with empty strings to prevent string-matching errors
    df_clean['title'] = df_clean['title'].fillna("")
    df_clean['news'] = df_clean['news'].fillna("")
    
    # 2. The Data-Enriched Taxonomy Mapping
    taxonomy = {
        # Religious Identities
        "Religious_Muslim": ['মুসলিম', 'ইসলাম', 'মুসলমান', 'ইমাম', 'মসজিদ', 'মসজিদে'],
        "Religious_Hindu": ['হিন্দু', 'পূজা', 'সংখ্যালঘু হিন্দু', 'মণ্ডপ', 'সংখ্যালঘু', 'সংখ্যালঘুদের'],
        "Religious_Others": ['বৌদ্ধ', 'খ্রিস্টান', 'গির্জা'],
        "Religious_Ahmadiyya": ['আহমদিয়া', 'কাদিয়ানী', 'আহমদিয়া'],
        "Religious_Baul": ['বাউল', 'মানবধর্ম', 'লালন', 'আবুল সরকার'],
        
        # Political-Ideological Identities
        "Political_AwamiLeague": ['আওয়ামী', 'লীগ', 'ক্ষমতাসীন', 'ছাত্রলীগ', 'আ.লীগ', 'আ.লীগের', 'লীগের', 'ফ্যাসিবাদ', 'ফ্যাসিস্ট', 'হাসিনা', 'স্বৈরাচার', 'পতিত সরকার'],
        "Political_BNP": ['বিএনপি', 'ছাত্রদল', 'বিএনপির', 'যুবদল', 'খালেদা জিয়া', 'তারেক রহমান'],
        "Political_Jamaat": ['জামায়াতে ইসলামি', 'শিবির', 'জামায়াতে', 'রাজাকার', 'আলবদর', 'আল শামস', 'জামায়াতের'],
        "Political_Conservative_Islamist": ['হেফাজত', 'হেফাজতে', 'হেফাজতের', 'ইসলামী আন্দোলন', 'আলেম', 'হুজুর', 'মাদ্রাসা', 'মাদরাসা', 'মাদরাসায়', 'মসজিদ', 'কওমি'], # Separated from generic Muslim
        "Political_NCP": ['জাতীয় নাগরিক পার্টি', 'এনসিপি', 'জাতীয় নাগরিক পার্টির', 'এনসিপির', 'NCP'],
        "Political_Left": ['বাম', 'বামপন্থী', 'তৃণমূল বাম', 'তৃণমূল বামপন্থী', 'তৃণমূল বামপন্থীদের', 'ছাত্র ইউনিয়ন', 'সর্বহারা', 'মাওবাদী', 'কমিউনিস্ট', 'সিপিবি', 'বাসদ'],
   
        # Demographic Identities
        "Demographic_Pahari": ['পাহাড়ি', 'আদিবাসী', 'ক্ষুদ্র নৃগোষ্ঠী', 'চট্টগ্রামের পাহাড়', 'চট্টগ্রামের পাহাড়ি', 'চট্টগ্রামের পাহাড়িদের', 'চাকমা', 'মারমা', 'সাঁওতাল', 'পাহাড়ী'],
        "Demographic_Rohingya": ['রোহিঙ্গা', 'রোহিঙ্গাদের', 'রোহিঙ্গা শরণার্থী', 'রোহিঙ্গা ক্যাম্প', 'রোহিঙ্গা জনগোষ্ঠী', 'উখিয়া', 'কুতুপালং', 'কুতুপালং শরণার্থী ক্যাম্প'],
        
        # Institutional Identities
        "Inst_Government": ['সরকার', 'সরকারি', 'মন্ত্রিপরিষদ', 'মন্ত্রীরা', 'প্রধানমন্ত্রী', 'মন্ত্রীর', 'মন্ত্রিপরিষদের'],
        "Inst_Military": ['সেনা', 'সেনাবাহিনী', 'সেনাবাহিনীর', 'বিমান বাহিনী', 'মেজর', 'সামরিক বাহিনী'],
        "Inst_Law_Enforcement": ['পুলিশ', 'র‍্যাব', 'আইনশৃঙ্খলা বাহিনী', 'আটক', 'গুম', 'পুলিশের', 'থানা', 'ডিবি', 'সিআইডি', 'সিটিটিসি', 'CTTC'],
        "Inst_Judiciary": ['আদালত', 'আদালতে', 'আদালতের', 'বিচারপতি', 'বিচারক', 'দুদক', 'দুদকের'],
        
        # Issue-Specific / Framing Identities
        "Frame_Quata_Reformist": ['কোটা', 'আন্দোলনকারী', 'বৈষম্যবিরোধী', 'ছাত্র-জনতার', 'বৈষম্যবিরোধী ছাত্র আন্দোলনকারী', 'জুলাই যোদ্ধা', 'ইনকিলাব', 'পুসাব', 'আন্দোলনে', 'আন্দোলনের', 'সমন্বয়ক'],
        "Frame_Islamist": ['ইসলামপন্থী', 'ইসলামপন্থি', 'কট্টর', 'রক্ষণশীল' 'জঙ্গি', 'উগ্রপন্থী', 'তৌহিদি জনতা', 'তাওহিদি জনতা', 'সন্ত্রাসী', 'সন্ত্রাসবাদী', 'সন্ত্রাসের', 'সন্ত্রাসীদের', 'মৌলবাদী', 'মৌলবাদী জনতা', 'মৌলবাদী আন্দোলনকারী', 'মব', 'ওয়ার', 'টেরর', 'চরমপন্থী', 'চরমপন্থি', 'চরমপন্থীদের', 'ইসলামবিদ্বেষ', 'দাড়ি-টুপি', 'টুপি-দাড়ি', 'দাড়ি-টুপি-ধারী', 'দাড়ি-টুপি-ধারীদের', 'তৌহিদি জনতার'],
        "Frame_Secular": ['প্রগতিশীল', 'মুক্তচিন্তা', 'মুক্তমনা', 'সুশীল', 'নাস্তিক', 'ধর্মনিরপেক্ষ', 'ধর্মনিরপেক্ষতাবাদী', 'সেক্যুলার', 'সেক্যুলারিজম'],
        "Frame_Protesters": ['বিক্ষোভকারী', 'প্রতিবাদকারী', 'বিক্ষোভকারীদের', 'ধর্মঘটকারী', 'বিক্ষোভের', 'বিক্ষোভে', 'বিক্ষোভে অংশগ্রহণকারী'],
        "Frame_Victims": ['ভুক্তভোগী', 'ক্ষতিগ্রস্ত', 'নিহত', 'আহত', 'গুম', 'গুমের', 'হত্যার', 'গণপিটুনি', 'নির্যাতন', 'হামলা', 'হামলার'],
        "Frame_Drug_Users": ['মাদক', 'মাদকাসক্ত', 'মাদক ব্যবসায়ী', 'মাদক কারবারি', 'মাদক সেবী', 'ইয়াবা', 'ফেনসিডিল', 'হেরোইন', 'কোকেন'],
    
        # Geopolitical Entities 
        "Geopolitical_India": ['ভারত', 'ভারতপন্থী', 'দিল্লি', 'নয়া দিল্লি', 'ভারতীয়', 'ইন্ডিয়া', 'পুশ ইন', 'বিএসএফ', 'সীমান্ত', 'সীমান্তে', 'চোরাকারবারি', ],
        "Geopolitical_West": ['আমেরিকা', 'মার্কিন', 'যুক্তরাষ্ট্র', 'পশ্চিমারা', 'পশ্চিম', 'ইউরোপীয় ইউনিয়ন', 'ওয়াশিংটন', 'স্যাঙ্কশন', 'ভিসা নীতি'],
        "Geopolitical_International": ['জাতিসংঘ', 'UN', 'বিশ্ব স্বাস্থ্য সংস্থা', 'বিশ্ব ব্যাংক', 'আইএমএফ', 'বিশ্ব বাণিজ্য সংস্থা', 'বিশ্ব মানবাধিকার সংস্থা'],
        "Geopolitical_Israel" : ['ইসরাইল', 'ইহুদী', 'জায়নিস্ট', 'যায়নিস্ট'],
        "Geopolitical_Palestine": ['জেরুজালেম', 'ফিলিস্তিন', 'হামাস', 'ফাতাহ'],
        "Geopolitical_Pakistan": ['পাকিস্তান', 'পাকিস্তানি', 'ইসলামাবাদ', 'পিন্ডি', 'রাওয়ালপিন্ডি'],
        "Geopolitical_China_Russia": ['চীন', 'বেজিং', 'রাশিয়া', 'মস্কো'],
        
        # Social & Professional Focus Areas
        "Social_Students_Academia": ['শিক্ষার্থী', 'ছাত্র', 'ছাত্রী', 'বিশ্ববিদ্যালয়', 'কলেজ', 'শিক্ষার্থীদের', 'শিক্ষার্থীরা', 'মেডিকেল শিক্ষার্থীদের', 'ইন্টার্ন চিকিৎসক', 'শিক্ষক'],
        "Social_Workers": ['শ্রমিক', 'দিনমজুর', 'গার্মেন্টস', 'পোশাক শ্রমিক'],
        "Social_Vulnerable": ['সুবিধাবঞ্চিত', 'বস্তিবাসী', 'শিশু'],
        "Social_Intellectuals": ['বুদ্ধিজীবী', 'সুশীল সমাজ', 'সুশীল', 'বিশিষ্ট নাগরিক', 'বিশ্লেষক'],
        "Social_Professions_Media": ['সাংবাদিক', 'সাংবাদিকরা', 'গণমাধ্যমকর্মী', 'প্রেস ক্লাব', 'প্রতিবেদক'],
        "Social_Professions_Legal": ['আইনজীবী', 'আইনজীবীরা', 'আইনজীবী সমিতি'],
        "Social_Business_Elites": ['ব্যবসায়ী', 'ব্যবসায়ীরা', 'সিন্ডিকেট', 'শিল্পপতি', 'কর্পোরেট', 'মালিকপক্ষ', 'পুঁজিবাদী']
    }
   
    
    # 3. Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    summary_counts = {}

    # 4. Process and export data
    master_output_path = os.path.join(output_dir, "master_separated_identities.xlsx")
    
    print("Processing identities and extracting subsets...\n")
    
    with pd.ExcelWriter(master_output_path, engine='openpyxl') as writer:
        
        for identity_name, keywords in taxonomy.items():
            # Create a regex pattern to match any of the keywords
            pattern = '|'.join(keywords)
            
            # Match pattern in either title or content
            mask = df_clean['title'].str.contains(pattern, na=False) | \
                   df_clean['news'].str.contains(pattern, na=False)
            
            filtered_df = df_clean[mask].copy()
            summary_counts[identity_name] = len(filtered_df)
            
            if len(filtered_df) > 0:
                # Excel sheet names have a strict 31 character limit
                sheet_name = identity_name[:31]
                
                # Write to the master workbook tab
                filtered_df.to_excel(writer, sheet_name=sheet_name, index=False)
                
                # Write to an individual standalone file
                individual_file_path = os.path.join(output_dir, f"{identity_name}.xlsx")
                filtered_df.to_excel(individual_file_path, index=False)
                
    # 5. Output Summary Report
    print("=== Extraction Summary ===")
    for identity, count in summary_counts.items():
        if count > 0:
            print(f"✔ {identity:<25} : {count} articles")
        else:
            print(f"✖ {identity:<25} : 0 articles (Skipped)")
            
    print("==========================")
    print(f"\nSuccess! All data has been separated and saved in the '{output_dir}' folder.")

# Execute the function on your local file
extract_enriched_identities("News_Amar_desh_cleaned.xlsx")

Loading dataset from: News_Amar_desh_cleaned.xlsx...
Processing identities and extracting subsets...

=== Extraction Summary ===
✔ Religious_Muslim          : 13540 articles
✔ Religious_Hindu           : 1672 articles
✔ Religious_Others          : 172 articles
✔ Religious_Ahmadiyya       : 6 articles
✔ Religious_Baul            : 244 articles
✔ Political_AwamiLeague     : 9159 articles
✔ Political_BNP             : 7558 articles
✔ Political_Jamaat          : 1381 articles
✔ Political_Conservative_Islamist : 4475 articles
✔ Political_NCP             : 1677 articles
✔ Political_Left            : 3369 articles
✔ Demographic_Pahari        : 351 articles
✔ Demographic_Rohingya      : 271 articles
✔ Inst_Government           : 13225 articles
✔ Inst_Military             : 5102 articles
✔ Inst_Law_Enforcement      : 15467 articles
✔ Inst_Judiciary            : 4486 articles
✔ Frame_Quata_Reformist     : 5873 articles
✔ Frame_Islamist            : 13715 articles
✔ Frame_Secular             : 67

In [1]:
import pandas as pd
import json

# Load CSV
df = pd.read_csv("plt_awami.csv")

output_file = "plt_awami.json"

data_list = []

for _, row in df.iterrows():

    title = str(row.get("News_Title", "")).strip()
    content = str(row.get("News_Content", "")).strip()

    # Merge safely
    if title and content:
        text = title + "\n\n" + content
    else:
        text = title or content

    task = {
        "data": {
            "text": text
        }
    }

    data_list.append(task)

# Write as JSON array
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data_list, f, ensure_ascii=False, indent=2)

print("Done ->", output_file)

Done -> plt_awami.json


In [16]:
import pandas as pd
import re


def filter_titles_by_keywords(
    input_file="prothom_alo.xlsx",
    title_column="title",
    output_csv="prothom_alo_keyword_matched.csv",
    check_content=False,
    content_column="news",
):
    """
    Keep rows where title contains any keyword.

    Optional:
    - Set check_content=True to also match keywords in content_column.

    Update the keywords list below with your static keywords.
    Saves matched rows to CSV and returns the matched DataFrame.
    """
    # Add your static keywords here
    keywords = [
        'বসুন্ধরা গ্রুপ', 'বসুন্ধরা গ্রুপের', 'মেঘনা গ্রুপ', 'মেঘনা গ্রুপের', 'এস আলম গ্রুপ', 'এস আলম গ্রুপের'
    ]

    if not keywords:
        raise ValueError("Please add at least one keyword to the keywords list.")

    df = pd.read_excel(input_file)

    if title_column not in df.columns:
        raise ValueError(
            f"Column '{title_column}' not found. Available columns: {list(df.columns)}"
        )

    if check_content and content_column not in df.columns:
        raise ValueError(
            f"check_content=True but column '{content_column}' was not found. "
            f"Available columns: {list(df.columns)}"
        )

    pattern = "|".join(re.escape(k) for k in keywords if str(k).strip())
    if not pattern:
        raise ValueError("Keywords list has no valid non-empty values.")

    title_series = df[title_column].fillna("").astype(str)
    title_mask = title_series.str.contains(pattern, case=False, na=False, regex=True)

    if check_content:
        content_series = df[content_column].fillna("").astype(str)
        content_mask = content_series.str.contains(pattern, case=False, na=False, regex=True)
        final_mask = title_mask | content_mask
    else:
        final_mask = title_mask

    matched_df = df[final_mask].copy()

    if output_csv:
        matched_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    return matched_df


# Example usage 1: title only
result = filter_titles_by_keywords(
    input_file="prothom_alo.xlsx",
    title_column="News_Title",
    output_csv="business_elites.csv",
)
print(f"Matched rows (title only): {len(result)}")

# Example usage 2: title + content (optional)
# result = filter_titles_by_keywords(
#     input_file="prothom_alo.xlsx",
#     title_column="News_Title",
#     output_csv="news_reporter.csv",
#     check_content=True,
#     content_column="News_Content",
# )
# print(f"Matched rows (title + content): {len(result)}")

Matched rows (title only): 5
